# EdgeFace-XS Fine-Tune on Indian Demographics

Fine-tunes pretrained EdgeFace-XS (edgenext_x_small + LoRA rank=0.6) on Indian face datasets.
Freezes backbone, fine-tunes last 2 blocks + ArcFace head.

**Run on:** Google Colab (T4 GPU) — ~6 hrs for 30 epochs

**Dataset:** Merged IndicFairFace + IMFDB + JFAD (prepared by `seed_indian_dataset.py`)

In [18]:
# Install dependencies
!pip install -q torch torchvision timm pillow

In [19]:
# Mount Google Drive for dataset + checkpoints
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = '/content/drive/MyDrive/nhai_face/dataset'
CHECKPOINT_DIR = '/content/drive/MyDrive/nhai_face/checkpoints'
PRETRAINED_PATH = '/content/drive/MyDrive/nhai_face/edgeface_xs_gamma_06.pt'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import math
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [21]:
# --- EdgeFace architecture (from edgeface repo) ---

class LoRaLin(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=True):
        super().__init__()
        self.linear1 = nn.Linear(in_features, rank, bias=False)
        self.linear2 = nn.Linear(rank, out_features, bias=bias)

    def forward(self, x):
        return self.linear2(self.linear1(x))


def replace_linear_with_lowrank(model, rank_ratio=0.2):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear) and 'head' not in name:
            in_f, out_f = module.in_features, module.out_features
            rank = max(2, int(min(in_f, out_f) * rank_ratio))
            setattr(model, name, LoRaLin(in_f, out_f, rank, bias=module.bias is not None))
        else:
            replace_linear_with_lowrank(module, rank_ratio)
    return model


def build_edgeface_xs():
    model = timm.create_model('edgenext_x_small')
    model.reset_classifier(512)
    model = replace_linear_with_lowrank(model, rank_ratio=0.6)
    return model

In [22]:
# --- ArcFace head ---

class ArcFaceHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, s: float = 64.0, m: float = 0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.empty(num_classes, feat_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - cosine.pow(2).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.unsqueeze(1), 1)
        output = one_hot * phi + (1.0 - one_hot) * cosine
        return output * self.s

In [23]:
# --- Data loading with augmentation ---
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

class CSVFaceDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, class_to_idx=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.img_col = self.df.columns[0]
        self.label_col = self.df.columns[-1]
        if class_to_idx is None:
            self.classes = sorted(self.df[self.label_col].astype(str).unique())
            self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        else:
            self.class_to_idx = class_to_idx
            self.classes = list(class_to_idx.keys())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, str(row[self.img_col]))
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.class_to_idx[str(row[self.label_col])]
        return img, label


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(112, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(112),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Load train.csv
full_df = pd.read_csv(os.path.join(DATASET_ROOT, 'train.csv'))
print('train.csv preview:')
print(full_df.head())
print(f'Total rows: {len(full_df)}')

# Drop singleton classes (can't stratify with 1 sample)
label_col = full_df.columns[-1]
counts = full_df[label_col].value_counts()
valid_classes = counts[counts >= 2].index
filtered_df = full_df[full_df[label_col].isin(valid_classes)].reset_index(drop=True)
print(f'After filtering singletons: {len(filtered_df)} rows, {filtered_df[label_col].nunique()} classes')

# 90/10 stratified split — same train/ folder for both
train_df, val_df = train_test_split(
    filtered_df, test_size=0.1, stratify=filtered_df[label_col], random_state=42,
)

train_img_dir = os.path.join(DATASET_ROOT, 'train')
train_ds = CSVFaceDataset(train_df, train_img_dir, transform=train_transform)
val_ds = CSVFaceDataset(val_df, train_img_dir, transform=val_transform,
                        class_to_idx=train_ds.class_to_idx)

NUM_CLASSES = len(train_ds.classes)
print(f'Classes: {NUM_CLASSES}, Train: {len(train_ds)}, Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)


train.csv preview:
          ID   Class
0    377.jpg  MIDDLE
1  17814.jpg   YOUNG
2  21283.jpg  MIDDLE
3  16496.jpg   YOUNG
4   4487.jpg  MIDDLE
Total rows: 19906
After filtering singletons: 19906 rows, 3 classes
Classes: 3, Train: 17915, Val: 1991


In [24]:
# --- Build model + load pretrained weights ---

backbone = build_edgeface_xs()
state_dict = torch.load(PRETRAINED_PATH, map_location='cpu')

# Strip "model." prefix from all keys (checkpoint was saved with a wrapper)
new_state_dict = {k.replace('model.', '', 1) if k.startswith('model.') else k: v
                  for k, v in state_dict.items()}

missing, unexpected = backbone.load_state_dict(new_state_dict, strict=False)
print(f'Missing keys: {len(missing)}, Unexpected: {len(unexpected)}')
if missing:    print('  Missing sample:', missing[:3])
if unexpected: print('  Unexpected sample:', unexpected[:3])

backbone = backbone.to(device)

arc_head = ArcFaceHead(512, NUM_CLASSES).to(device)

# Freeze everything except last 2 stages + classifier head
for name, param in backbone.named_parameters():
    param.requires_grad = False

# Unfreeze last 2 stages (stages.2, stages.3 in edgenext) + head
for name, param in backbone.named_parameters():
    if 'stages.2.' in name or 'stages.3.' in name or 'head' in name or 'norm' in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')


Missing keys: 0, Unexpected: 0
Trainable: 1,643,928 / 1,770,492 (92.9%)


In [25]:
# --- Training setup ---

EPOCHS = 30
LR = 1e-4

optimizer = torch.optim.AdamW(
    list(filter(lambda p: p.requires_grad, backbone.parameters())) + list(arc_head.parameters()),
    lr=LR, weight_decay=0.05,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

In [27]:
# --- Training loop ---
from torch.amp import autocast, GradScaler

scaler = GradScaler('cuda')
best_val_acc = 0.0

# Collect trainable params for grad clipping
trainable_params = [p for p in list(backbone.parameters()) + list(arc_head.parameters())
                    if p.requires_grad]

for epoch in range(EPOCHS):
    backbone.train()
    arc_head.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with autocast('cuda'):
            embeddings = backbone(images)
            logits = arc_head(embeddings, labels)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    scheduler.step()
    train_acc = correct / total
    avg_loss = total_loss / total

    # --- Validation ---
    backbone.eval()
    arc_head.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with autocast('cuda'):
                embeddings = backbone(images)
                logits = arc_head(embeddings, labels)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total += images.size(0)
    val_acc = val_correct / val_total

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | loss={avg_loss:.4f} | '
          f'train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | '
          f'lr={scheduler.get_last_lr()[0]:.6f}')

    # --- Save last checkpoint (overwrite) ---
    last_path = os.path.join(CHECKPOINT_DIR, 'edgeface_xs_indian_last.pt')
    torch.save({
        'epoch': epoch + 1,
        'backbone': backbone.state_dict(),
        'arc_head': arc_head.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_acc': val_acc,
    }, last_path)

    # --- Save best checkpoint ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_path = os.path.join(CHECKPOINT_DIR, 'edgeface_xs_indian_best.pt')
        torch.save(backbone.state_dict(), best_path)
        print(f'  -> New best: {val_acc:.4f}, saved to {best_path}')

print(f'\nTraining complete. Best val_acc: {best_val_acc:.4f}')


Epoch 01/30 | loss=18.6302 | train_acc=0.2556 | val_acc=0.2838 | lr=0.000100
  -> New best: 0.2838, saved to /content/drive/MyDrive/nhai_face/checkpoints/edgeface_xs_indian_best.pt
Epoch 02/30 | loss=12.4020 | train_acc=0.3891 | val_acc=0.3908 | lr=0.000099
  -> New best: 0.3908, saved to /content/drive/MyDrive/nhai_face/checkpoints/edgeface_xs_indian_best.pt
Epoch 03/30 | loss=9.3755 | train_acc=0.4044 | val_acc=0.2757 | lr=0.000098
Epoch 04/30 | loss=8.2108 | train_acc=0.3710 | val_acc=0.3184 | lr=0.000096
Epoch 05/30 | loss=8.2149 | train_acc=0.3671 | val_acc=0.2220 | lr=0.000093
Epoch 06/30 | loss=7.8014 | train_acc=0.3833 | val_acc=0.2742 | lr=0.000090
Epoch 07/30 | loss=7.2596 | train_acc=0.4191 | val_acc=0.3159 | lr=0.000087
Epoch 08/30 | loss=6.7009 | train_acc=0.4620 | val_acc=0.2803 | lr=0.000083
Epoch 09/30 | loss=6.2704 | train_acc=0.4895 | val_acc=0.2707 | lr=0.000079
Epoch 10/30 | loss=5.7941 | train_acc=0.5250 | val_acc=0.2742 | lr=0.000075
Epoch 11/30 | loss=5.5706 | tr

In [31]:
# --- Export best model to ONNX ---
!pip install -q onnx onnxruntime onnxscript

import onnx
import onnxruntime as ort
import numpy as np

# Load best checkpoint
backbone.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, 'edgeface_xs_indian_best.pt'), map_location='cpu')
)
backbone.eval().cpu()

dummy = torch.randn(1, 3, 112, 112)
onnx_path = os.path.join(CHECKPOINT_DIR, 'edgeface_xs_indian.onnx')

# Use legacy exporter (dynamo=False) — more stable for older opsets
torch.onnx.export(
    backbone, dummy, onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=13,
    do_constant_folding=True,
    dynamo=False,
)

# --- Verify export ---
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

with torch.no_grad():
    torch_out = backbone(dummy).numpy()

ort_sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
onnx_out = ort_sess.run(None, {'input': dummy.numpy()})[0]

max_diff = np.abs(torch_out - onnx_out).max()
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

print(f'ONNX exported to: {onnx_path}')
print(f'Size: {size_mb:.2f} MB')
print(f'Output shape: {onnx_out.shape}')
print(f'Max diff (PyTorch vs ONNX): {max_diff:.2e}')
assert max_diff < 1e-4, 'PyTorch and ONNX outputs diverge — check export!'
print('✅ Verification passed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.9 MB/s eta 0:00:00


/tmp/ipykernel_3976/1922850167.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX exported to: /content/drive/MyDrive/nhai_face/checkpoints/edgeface_xs_indian.onnx
Size: 6.97 MB
Output shape: (1, 512)
Max diff (PyTorch vs ONNX): 5.22e-08
✅ Verification passed
